In [2]:
from selenium import webdriver
import time
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [3]:
movies = []
review_links = []
base_url = "https://www.momo.vn/cinema"
url = base_url + "/review"

In [4]:
print(f"Đang bắt đầu crawl dữ liệu từ: {url}")
driver = webdriver.Chrome()
driver.get(url)
wait = WebDriverWait(driver, 15)
more_btn = driver.find_element(By.CSS_SELECTOR,"button.border-pink-600")
print(more_btn)

Đang bắt đầu crawl dữ liệu từ: https://www.momo.vn/cinema/review
<selenium.webdriver.remote.webelement.WebElement (session="684829689616301a4d7944ff9b8c2897", element="f.AC169767FF46CE169E98E353A0E59094.d.1F30B97198303C44B5A0A00A287C8811.e.32")>


# Xử lý lấy link các phim

In [42]:
loops = 120

In [ ]:
# Xử lý nút xem thêm để lấy list các film
for i in range(loops):
    try:
        more_btn = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "button.border-pink-600")))
        driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", more_btn)
        time.sleep(1.5) 
        driver.execute_script("arguments[0].click();", more_btn)
        print(f"--- Đã bấm 'Xem thêm' lần {i+1} ---")
        time.sleep(2) 
        
    except Exception:
        print("Đã load hết dữ liệu hoặc không tìm thấy nút 'Xem thêm' nữa.")
        break

--- Đã bấm 'Xem thêm' lần 1 ---
--- Đã bấm 'Xem thêm' lần 2 ---
--- Đã bấm 'Xem thêm' lần 3 ---
--- Đã bấm 'Xem thêm' lần 4 ---
--- Đã bấm 'Xem thêm' lần 5 ---
--- Đã bấm 'Xem thêm' lần 6 ---
--- Đã bấm 'Xem thêm' lần 7 ---
--- Đã bấm 'Xem thêm' lần 8 ---
--- Đã bấm 'Xem thêm' lần 9 ---
--- Đã bấm 'Xem thêm' lần 10 ---
--- Đã bấm 'Xem thêm' lần 11 ---
--- Đã bấm 'Xem thêm' lần 12 ---
--- Đã bấm 'Xem thêm' lần 13 ---
--- Đã bấm 'Xem thêm' lần 14 ---
--- Đã bấm 'Xem thêm' lần 15 ---
--- Đã bấm 'Xem thêm' lần 16 ---
--- Đã bấm 'Xem thêm' lần 17 ---
--- Đã bấm 'Xem thêm' lần 18 ---
--- Đã bấm 'Xem thêm' lần 19 ---
--- Đã bấm 'Xem thêm' lần 20 ---
--- Đã bấm 'Xem thêm' lần 21 ---
--- Đã bấm 'Xem thêm' lần 22 ---
--- Đã bấm 'Xem thêm' lần 23 ---
--- Đã bấm 'Xem thêm' lần 24 ---
--- Đã bấm 'Xem thêm' lần 25 ---
--- Đã bấm 'Xem thêm' lần 26 ---
--- Đã bấm 'Xem thêm' lần 27 ---
--- Đã bấm 'Xem thêm' lần 28 ---
--- Đã bấm 'Xem thêm' lần 29 ---
--- Đã bấm 'Xem thêm' lần 30 ---
--- Đã bấm 'Xem thê

In [ ]:
# Xử lý chữ xem thêm để lấy link
review_elements = driver.find_elements(By.XPATH, "//a[contains(., 'Xem thêm')]")
for i in review_elements:
    link = i.get_attribute("href")
    if link not in review_links: # Tránh lấy trùng
        review_links.append(link)
print(f"Đã tìm thấy tổng cộng {len(review_links)} đường dẫn review.")
for i, link in enumerate(review_links[:5]): # In thử 5 link đầu
    print(f"{i+1}. {link}")

Đã tìm thấy tổng cộng 366 đường dẫn review.
1. https://www.momo.vn/cinema/tho-oi-24729/review
2. https://www.momo.vn/cinema/avatar-fire-and-ash-24650/review
3. https://www.momo.vn/cinema/bau-vat-troi-cho-24696/review
4. https://www.momo.vn/cinema/nha-ba-toi-mot-phong-24748/review
5. https://www.momo.vn/cinema/tai-24788/review


In [ ]:
import csv 
# Chuyển link vào file csv
file_name = "link_list_momo.csv"
with open(file_name,mode='w',newline='',encoding='utf-8-sig') as file:
    writer = csv.writer(file)
    writer.writerow(['Review_link'])
    for link in review_links:
        writer.writerow([link])
print(f"Đã xuất thành công {len(review_links)} link vào file {file_name}")

Đã xuất thành công 366 link vào file link_list_momo.csv


# Xử lý trích xuất metadata từng phim

In [10]:
import pandas as pd
import os
def extract_movie_metadata(container_element):
    metadata = {}
    try:
        # Tên phim
        metadata['title'] = container_element.find_element(By.CSS_SELECTOR, "div.font-bold").text.strip()
        
        # Thể loại, ngày xuất bản, quốc gia
        items = container_element.find_elements(By.TAG_NAME, "li")
        metadata['genre'] = "N/A"
        metadata['release_date'] = "N/A"
        metadata['country'] = "N/A"
        
        for li in items:
            text = li.text
            if "Thể loại" in text:
                metadata['genre'] = text.split(":")[-1].strip()
            elif "Ngày chiếu" in text:
                metadata['release_date'] = text.split(":")[-1].strip()
            elif "Quốc gia" in text:
                metadata['country'] = text.split(":")[-1].strip()
                
    except Exception as e:
        print(f"Lỗi khi trích xuất metadata: {e}")
    return metadata

def save_to_csv(all_movies_data, reviews_data, url):
    if not all_movies_data: return
    movie_title = all_movies_data[0]['title']
    
    # Lưu Metadata
    df_meta = pd.DataFrame(all_movies_data)
    df_meta.to_csv('movies_metadata.csv', mode='a', index=False, 
                   header=not os.path.exists('movies_metadata.csv'), encoding='utf-8-sig')

    # Lưu Reviews
    # for r in reviews_data:
    #     r['Movie_Title'] = movie_title
    #     r['Movie_URL'] = url
    # df_reviews = pd.DataFrame(reviews_data)
    # df_reviews.to_csv('movie_reviews.csv', mode='a', index=False, 
    #                   header=not os.path.exists('movie_reviews.csv'), encoding='utf-8-sig')
    # print(f"Đã xong: {movie_title} ({len(reviews_data)} reviews)")

# Pipeline đầy đủ trích xuất film và metadata

In [ ]:
df_links = pd.read_csv('link_list_momo.csv')
urls = df_links['Review_link'].tolist()
print(urls)
for current_url in urls:
    print(f"\nĐang truy cập: {current_url}")
    all_movies_data = []
    reviews_data = []
    
    try:
        driver.get(current_url)
        time.sleep(2) # Đợi trang ổn định
        
        # 2. Lấy Metadata 
        try:
            movie_info = extract_movie_metadata(driver)
            movie_info['url'] = current_url
            all_movies_data.append(movie_info)
        except:
            print("Không lấy được metadata, bỏ qua phim này.")
            continue

        # 3. Vòng lặp "Xem tiếp nhé!"
        loops = 50 # Bạn có thể điều chỉnh số lần bấm tùy ý
        for i in range(loops):
            try:
                more_btns = driver.find_elements(By.XPATH, "//button[contains(., 'Xem tiếp nhé!')]")
                if len(more_btns) == 0:
                    print(f"--- Đã hết bình luận tại lần scroll thứ {i}, chuyển sang trích xuất... ---")
                    break
                more_btn = more_btns[0]
                driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", more_btn)
                print(f"Đang scroll lần thứ {i+1}")
                time.sleep(1.5)
                driver.execute_script("arguments[0].click();", more_btn)
                time.sleep(1.25)
            except:
                break

        # 4. Mở rộng "...Xem thêm" bình luận
        read_more_btns = driver.find_elements(By.XPATH, "//span[contains(text(), '...Xem thêm')]")
        for btn in read_more_btns:
            try: driver.execute_script("arguments[0].click();", btn)
            except: continue
        
        # 5. Trích xuất Review
        review_items = driver.find_elements(By.CSS_SELECTOR, "div.relative.mt-3")
        for item in review_items:
            try:
                comment_text = item.find_element(By.CSS_SELECTOR, "div.leading-relaxed").text.strip()
                rating_element = item.find_element(By.XPATH, ".//span[contains(@class, 'pl-0.5')]")
                rating_raw = rating_element.text.strip()
                
                reviews_data.append({
                    "rating": rating_raw.split("/")[0],
                    "comment": comment_text
                })
            except:
                continue

        # 6. Lưu vào CSV ngay lập tức sau mỗi phim để tránh mất dữ liệu
        save_to_csv(all_movies_data, reviews_data, current_url)

    except Exception as e:
        print(f"Lỗi tại {current_url}: {e}")

['https://www.momo.vn/cinema/tho-oi-24729/review', 'https://www.momo.vn/cinema/avatar-fire-and-ash-24650/review', 'https://www.momo.vn/cinema/bau-vat-troi-cho-24696/review', 'https://www.momo.vn/cinema/nha-ba-toi-mot-phong-24748/review', 'https://www.momo.vn/cinema/tai-24788/review', 'https://www.momo.vn/cinema/cam-on-nguoi-da-thuc-cung-toi-24635/review', 'https://www.momo.vn/cinema/tieu-yeu-quai-nui-lang-lang-24760/review', 'https://www.momo.vn/cinema/mui-pho-24741/review', 'https://www.momo.vn/cinema/nha-minh-di-thoi-24789/review', 'https://www.momo.vn/cinema/quy-nhap-trang-2-24791/review', 'https://www.momo.vn/cinema/mud-born-24738/review', 'https://www.momo.vn/cinema/goat-24636/review', 'https://www.momo.vn/cinema/wuthering-heights-24633/review', 'https://www.momo.vn/cinema/hoppers-24647/review', 'https://www.momo.vn/cinema/pets-on-a-train-24790/review', 'https://www.momo.vn/cinema/once-we-were-us-24795/review', 'https://www.momo.vn/cinema/chickenhare-and-the-secret-of-the-groundho

In [16]:
def save_to_csv(all_movies_data):
    if not all_movies_data: 
        return
    
    # 1. Chuyển list metadata thành DataFrame
    df_meta = pd.DataFrame(all_movies_data)
    
    # 2. Đường dẫn file lưu
    file_name = 'movies_metadata.csv'
    
    # 3. Ghi vào CSV (mode='a' để ghi tiếp, header chỉ ghi lần đầu)
    df_meta.to_csv(file_name, mode='a', index=False, 
                   header=not os.path.exists(file_name), 
                   encoding='utf-8-sig')
    
    print(f"Đã lưu metadata phim: {all_movies_data[0].get('title', 'N/A')}")

In [17]:
import pandas as pd
df = pd.read_csv('link_list_momo.csv')
urls = df['Review_link'].tolist()
for url in urls:
    try:
        driver.get(url)
        time.sleep(1)
        movie_info = extract_movie_metadata(driver)
        movie_info['url'] = url
        
        # Tạo list chứa 1 dict phim hiện tại để truyền vào hàm
        current_movie_list = [movie_info] 
        
        # Chỉ truyền list metadata vào hàm
        save_to_csv(current_movie_list)
        
    except Exception as e:
        print(f"Lỗi: {e}")
    
    

Đã lưu metadata phim: Thỏ Ơi!!
Đã lưu metadata phim: Avatar: Lửa và Tro Tàn
Đã lưu metadata phim: Báu Vật Trời Cho
Đã lưu metadata phim: Nhà Ba Tôi Một Phòng
Đã lưu metadata phim: TÀI
Đã lưu metadata phim: Cảm Ơn Người Đã Thức Cùng Tôi
Đã lưu metadata phim: Tiểu Yêu Quái Núi Lãng Lãng
Đã lưu metadata phim: Mùi Phở
Đã lưu metadata phim: Nhà Mình Đi Thôi
Đã lưu metadata phim: Quỷ Nhập Tràng 2
Đã lưu metadata phim: Kumanthong Đài Loan: Oán Thai Đòi Mẹ
Đã lưu metadata phim: Tuyển Thủ Dê: Mùi Vị Chiến Thắng
Đã lưu metadata phim: Đồi Gió Hú
Đã lưu metadata phim: Hoppers: Cú Nhảy Kỳ Diệu
Đã lưu metadata phim: Biệt Đội Thú Cưng: Cuộc Chiến Trên Đường Ray
Đã lưu metadata phim: Không Còn Chúng Ta
Đã lưu metadata phim: Thỏ Gà Du Xuân: Đại Náo Địa Đạo
Đã lưu metadata phim: Quốc Bảo
Đã lưu metadata phim: Khủng Long Đón Tết
Đã lưu metadata phim: Chuyện Kinh Dị Giới Siêu Giàu
Đã lưu metadata phim: Lời Nguyền Thư Viện Mikura
Đã lưu metadata phim: Huyền Tình Dạ Trạch
Đã lưu metadata phim: Stray K

In [18]:
import pandas as pd

# 1. Đọc file kết quả metadata mà bạn vừa lưu
try:
    df_result = pd.read_csv('movies_metadata.csv')

    # 2. Khởi tạo một set trống
    unique_genres = set()

    # 3. Lặp qua cột 'genre'
    # .dropna() để tránh lỗi nếu có hàng nào đó bị trống dữ liệu
    for genres_str in df_result['genre'].dropna():
        # Kiểm tra nếu dữ liệu là N/A (do hàm extract_metadata của bạn gán) thì bỏ qua
        if genres_str == "N/A":
            continue
            
        # Tách chuỗi "Chính kịch, Hài" thành list ["Chính kịch", "Hài"]
        genres_list = [g.strip() for g in genres_str.split(',')]
        
        # Thêm list này vào set (update sẽ tự động lọc trùng)
        unique_genres.update(genres_list)

    # 4. In kết quả để kiểm tra
    print(f"Đã trích xuất thành công {len(unique_genres)} thể loại duy nhất:")
    print(unique_genres)

except FileNotFoundError:
    print("Lỗi: Không tìm thấy file 'movies_metadata.csv'. Hãy kiểm tra lại quá trình lưu file.")

Đã trích xuất thành công 32 thể loại duy nhất:
{'Siêu anh hùng', 'Tôn Giáo', 'Gia đình', 'Tài liệu', 'Gay cấn', 'Lãng mạn', 'Cổ Trang', 'Truyền hình thực tế', 'Giả tưởng', 'Drama', 'Trinh thám', 'Ma', 'Hành động', 'Tâm lý', 'Tình cảm', 'Âm Nhạc', 'Bí ẩn', 'Chiến tranh', 'Chính kịch', 'Giật gân', 'Tâm Linh', 'Phiêu lưu', 'Đẹp Trai', 'Tội phạm', 'Khoa học - Viễn tưởng', 'Thể thao', 'Hoạt hình', 'Thảm hoạ', 'Hài', 'Kinh dị', 'Lịch sử', 'Hình sự'}
